In [69]:
""" import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))
"""

' import torch\nprint(torch.cuda.is_available(), torch.cuda.get_device_name(0))\n'

# Create and run a local RAG pipeline

## Goals for RAG

* Retrieval - Find relevant info given a query, e.g. "what are the macronutrients and what do they do?" -> retrieves passages of text realted to the macronutrients from a nutrition textbok.
* Augmented - Take the relevant info and augment our input(prompt) to an LLM with that relevant info
* Generation - Take the first two steps and pass them to an LLM for generative outputs

## Main Goal
improve the outputs of LLMs by:
1. Preventing hallucinations
2. Work with custom data


## Use Cases:
* Textbook Q&A - find releveant passages to the questions you have.

In [70]:
import os
import requests

# Get PDF document path
pdf_path = "sustainability-action-plan.pdf"

# Download PDF
if not os.path.exists(pdf_path):
    print(f"[INFO] File doesn't exist, downloading...")

    #Enter the URL of the PDF
    url = "https://cityofwinterpark.org/docs/departments/natural-resources-sustainability/sustainability-division/sustainability-action-plan/sustainability-action-plan-annual-report-2023.pdf"

    #The local filename to save the downloaded file
    filename=pdf_path

    # Browser-like User-Agent required: the site returns 403 for python-requests' default
    headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
}


    #Send a GET request to the URL
    response = requests.get(url, headers=headers, timeout=30)

    #Check if the request was successful
    if response.status_code ==200:
        # Open the file and save it
        with open(filename, "wb") as file:
            file.write(response.content)

        print(f"[INFO] The file has been downloaded and saved as {filename}")
    else:
        print(f"[INFO] Failed to download the file. Status code: {response.status_code}")

else:
    print(f"File {pdf_path} exists.")



File sustainability-action-plan.pdf exists.


In [71]:
%pip install tqdm
%pip install PyMuPDF

In [72]:
import pymupdf # requires: !pip install PyMuPDF
from tqdm.auto import tqdm

def text_formatter(text: str) -> str:
    """Performs minor formatting on text."""
    cleaned_text= text.replace("\n"," ").strip()

    return cleaned_text

PRINTED_PAGE_OFFSET = 2
def open_and_read_pdf(pdf_path: str) -> list[dict]:
    doc = pymupdf.open(pdf_path)
    pages_and_texts=[]
    
    for page_number, page in tqdm(enumerate(doc)):
        if page_number < PRINTED_PAGE_OFFSET:
            continue # skip the two pages before the title page
        
        text = page.get_text()
        text = text_formatter(text=text)
        pages_and_texts.append({"page_number": page_number - PRINTED_PAGE_OFFSET + 1,
                                "page_char_count": len(text),
                                "page_word_count": len(text.split(" ")),
                                "page_sentence_count": len(text.split(". ")),
                                "page_token_count": len(text) / 4, # 1 token = ~ 4 characters
                                "text": text})

    return pages_and_texts
pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]

0it [00:00, ?it/s]

[{'page_number': 1,
  'page_char_count': 406,
  'page_word_count': 114,
  'page_sentence_count': 1,
  'page_token_count': 101.5,
  'text': 'City of Winter Park  Sustainability Action Plan                  January 24th, 2024                            Presented by:   Gloria Eby, Natural Resources and Sustainability Director     Sara Miller, Sustainability Manager   Mia Brady, Sustainability Specialist   Keep Winter Park Beautiful & Sustainable Advisory Board           Certified WCAG 2.1 Level AA compliant and usable with JAWS and NVDA by SWH'},
 {'page_number': 2,
  'page_char_count': 2873,
  'page_word_count': 155,
  'page_sentence_count': 22,
  'page_token_count': 718.25,
  'text': '2    Contents   Background ........................................................................................................................................... 3  Vision and Purpose ......................................................................................................................

In [73]:
pages_and_texts[9]

{'page_number': 10,
 'page_char_count': 1771,
 'page_word_count': 277,
 'page_sentence_count': 4,
 'page_token_count': 442.75,
 'text': '10  o Parks and Recreation summer camp program incorporates Orange County Public Schools (OCPS) Summer Food Program o Summer Feeding - Orange County Public Schools (ocps.net) • Winter Park Library o Master gardening basics:  5 classes, partnership with UF/IFAS Extension orange County o Seed library: 3 grow seasons; 9,000 seeds distributed per year • Rollins College o Food pantry for students – Students can either come to the pantry and take what they need or place an order online, which will be processed at the C-Store and stored in a pre-packed bag o Sam Justice is staff contact - sjustice@rollins.edu o Urban farm collaboration - providing fresh produce to the food pantry on a regular basis • SeniorsFirst o https://www.seniorsfirstinc.org/ o Provides meals-on-wheels to seniors in Winter Park • Winter Park High School o Willies Mart – “Willie’s Mart p

In [74]:

df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count,page_token_count,text
0,1,406,114,1,101.50,City of Winter Park Sustainability Action Pla...
1,2,2873,155,22,718.25,2 Contents Background ......................
2,3,2448,420,12,612.00,3 Background Vision and Purpose The 2022 Su...
3,4,2954,505,16,738.50,"4 In 2012, the city’s Environmental Review an..."
4,5,2551,411,7,637.75,5 indicators to benchmark city status in effo...


## Table Preprocessing

The goal is to extract tables from the PDF while preserving the relationships between column headers and row values

In [75]:
doc = pymupdf.open(pdf_path)

for page_number, page in enumerate(doc):
    tables = page.find_tables().tables

    if tables:
        print(f"Page {page_number}: {len(tables)} table(s) found")

Page 5: 1 table(s) found
Page 6: 1 table(s) found
Page 12: 1 table(s) found
Page 13: 1 table(s) found
Page 14: 1 table(s) found
Page 15: 1 table(s) found
Page 18: 2 table(s) found
Page 19: 1 table(s) found
Page 20: 1 table(s) found
Page 22: 2 table(s) found
Page 23: 1 table(s) found
Page 25: 2 table(s) found
Page 26: 1 table(s) found
Page 27: 1 table(s) found
Page 28: 1 table(s) found
Page 31: 1 table(s) found
Page 32: 1 table(s) found
Page 33: 1 table(s) found
Page 37: 2 table(s) found
Page 38: 1 table(s) found
Page 39: 1 table(s) found
Page 40: 1 table(s) found
Page 42: 2 table(s) found
Page 43: 1 table(s) found
Page 45: 2 table(s) found
Page 46: 1 table(s) found
Page 47: 1 table(s) found
Page 51: 1 table(s) found
Page 52: 2 table(s) found
Page 55: 2 table(s) found
Page 57: 1 table(s) found


In [76]:
# Automatically extract every table

all_tables = []

for page_number, page in enumerate(doc):
    tables = page.find_tables().tables

    for table_index, table in enumerate(tables):
        table_df = table.to_pandas()

        all_tables.append({
            "page_number" : page_number,
            "table_index": table_index,
            "data": table_df
        })

print(f"Extracted {len(all_tables)} tables")

Extracted 39 tables


# Inspect pages with Tables
## Goals:
1. Turn the tables into dataframe to inspect the actual rows and columns instead of treating the table as one long string

In [77]:
# Inspect the first table
display(all_tables[0]["data"])

,2012-2013 KWPB&S Board Members,Col1,2014-2015 KWPB&S Board Members,Col3
0,"Mary Dipboye, Chair",James (Bob) Robinson,"Michael Poole, Chair",Mark Roush
1,"Stephen Pategas, V. Chair",Pat Schoknecht,"Stephen Pategas, V. Chair",Bruce Thomas
2,Michele Hipp,Julia Tensfeldt,Michele Hipp,Steven DiClemente
3,Michael Poole,Kent Tse,Raymond Randall,Mary Dipboye
4,Raymond Randall,Laura Walda,Pat Schoknecht,John Tapp
5,John Rife,Carol Kostick,Julia Tensfeldt,Fred Kosiewski
6,Lucy Roberts,Mark Roush,Laura Walda,Cathy Blanton
7,Joseph Robillard,,Carol Shenck (Kostick),


In [78]:
# Check which page it came from
print(all_tables[0]["page_number"])

5


In [79]:
display(all_tables[1]["data"])
print(all_tables[1]["page_number"])

,"health, sustainability and aesthetics r, more beautiful place to live, work,",of our environment in o and play.,order to create a
0,2022 KWPB&S Board Members,Appointed By,End of Term
1,"Ben Ellis, Vice Chair",Mayor Anderson,2024
2,Carey Bond,Commissioner DeCiccio,2023
3,Mark J Yonker,Commissioner Cruzada,2025
4,"Kay Hudson, Chair",Mayor Anderson,2024
5,Stephen Pategas,Commissioner Weaver,2025
6,Rishona S Teres,Commissioner Sullivan,2023
7,Laura Gustafson-Hullinger,Mayor Anderson,2024


6


In [80]:
display(all_tables[2]["data"])
print(all_tables[2]["page_number"])

,Col0,INDICATORS,Col2,Col3,Col4,Col5,Col6,Col7,Col8,Col9,Col10
0,,Indicator Description,2012\nBaseline,2021\nStatus,2022\nStatus,2025\nTarget,2035\nTarget,2042\nTarget,None,2050\nTarget,None
1,CR-1,Proportion of renewable energy in\nWinter Park...,4%,13%,21.72,23%,80%,89%,None,100%,None
2,CR-2,Community wide greenhouse gas\nemissions [Tons...,"398,919","405,394","406,999","355,000","235,000","151,000",None,"58,000",None
3,CR-3,WP Electric Utility customers with\nSolar – Ba...,7,139,154,300,700,950,None,1300,None
4,CR-4,Proportion of Residents within 1/2\nmile of af...,-,46%,40%,50%,75%,80%,None,TBD,None
5,CR-5,Undergrounding of Winter Park’s\nelectrical Ut...,-,-,73.1%,85%,100%,,,,


12


In [81]:
display(all_tables[6]["data"])
print(all_tables[6]["page_number"])

,Col0,Indicator Description,2023\nBaseline,2024\nTarget,2026\nTarget,2030\nTarget,2035\nTarget
0,ES-1,Annual renewable energy\ncontribution to the t...,21.72%,21.72%,21.72%,50%,80%
1,ES-2,City residential rates as\ncompared to FL publ...,<100%,<105%,<105%,<105%,<105%
2,ES-3,EE and DR program milestones,Not\napplicable,Complete studies\nand develop\nimplementation\...,Implement\nprograms\nfor\ncustomer\nEE and DR,,
3,ES-4,Milestones to implement\nchanges to rate to en...,Not\napplicable,Complete rate\nstudies and develop\nrate desig...,Implement\nnew rates as\nrecommend\ned by studies,,
4,ES-5,Electric Utility strategy and\ngrid analyses m...,,"Complete a strategic\nplan, distributed\nsolar...",,,


18


In [82]:
import random
random.sample(pages_and_texts, k=3)

[{'page_number': 45,
  'page_char_count': 1910,
  'page_word_count': 366,
  'page_sentence_count': 1,
  'page_token_count': 477.5,
  'text': '45  Projected  Implementation  Year  Action  Accomplished  Status  Responsible  Department(s)  2023  Maintain the list of the city’s  Registered Haulers    Yes  Ongoing  Sustainability  2023  Assist multi-family and commercial  buildings with creating recycling  education and outreach plans   No  No  Sustainability   Division,   Economic   Development  2023  Increase recycling opportunities at  city-owned public facilities and parks  Yes  Initiated  Parks and  Recreation,  Sustainability  2023  Consider food scrap collection and   Pay As You Throw options in Scope of  Work description for Solid Waste   Services Solicitation    No  No  City   Administration,  Sustainability   2025  Establishing an ordinance to ban all  Styrofoam products city-wide   No  No  City   Administration,  Sustainability   2026  Consider a mandate for commercial  ordinance

In [83]:
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count,page_token_count
count,57.0,57.00,57.00,57.00,57.00
mean,29.0,2050.11,353.60,9.28,512.53
std,16.6,701.39,113.62,7.93,175.35
min,1.0,406.00,114.00,1.00,101.50
25%,15.0,1559.00,279.00,2.00,389.75
50%,29.0,1958.00,366.00,7.00,489.50
75%,43.0,2420.00,420.00,15.00,605.00
max,57.0,3876.00,639.00,33.00,969.00


In [84]:
# Drop Heading and Table of contents
records_df = records_df[~records_df.page_number.isin([1, 2])].reset_index(drop=True)

Interpreting the token count
Caution:

1.  LLMs don't deal with infinite tokens
2. Embedding models don't deal with infinite tokens